# Junior Question Benchmark

This notebook runs the `turboquant-rag` pipeline over a user-defined set of questions from `questions.json` and records the profiling metrics used in the Junior profiling experiment:

- TTFT: Time To First Token
- TPOT: Time Per Output Token
- Peak VRAM from `torch.cuda`
- Peak VRAM and utilization from `nvidia-smi`
- Optional answer metrics when `reference_answer` is present: exact match and token F1

The benchmark runner lives in `rag_library.benchmarks`.

## 1. Configuration

Edit `QUESTION_LIMIT` to run the first N questions from `questions.json`.
Set `BACKEND` to `turboquant` for the compressed KV-cache run, or `bf16` for the baseline if your GPU has enough memory.

In [ ]:
from pathlib import Path
import json
import os
import sys

def find_repo_root(start: Path) -> Path:
    candidates = [start, *start.parents]
    for candidate in candidates:
        if (candidate / 'rag_library').exists():
            return candidate
    return start

REPO_ROOT = find_repo_root(Path.cwd())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

QUESTIONS_PATH = REPO_ROOT / 'questions.json'
DATA_DIR = REPO_ROOT / 'data'
RESULTS_DIR = REPO_ROOT / 'results' / 'question_benchmark'
INDEX_PATH = RESULTS_DIR / 'rag_index.faiss'
CHUNKS_PATH = RESULTS_DIR / 'rag_chunks.json'
CORPUS_PATH = DATA_DIR / 'corpus.json'

QUESTION_LIMIT = 5
BACKEND = 'turboquant'  # 'turboquant' or 'bf16'
BIT_WIDTH = 3.5
MAX_NEW_TOKENS = 64
USE_PROFILER = False
RUN_NAME = None

print('REPO_ROOT:', REPO_ROOT)
print('QUESTIONS_PATH:', QUESTIONS_PATH)
print('CORPUS_PATH:', CORPUS_PATH)
print('RESULTS_DIR:', RESULTS_DIR)


## 2. Load Environment and Build the Model

This notebook assumes your `.env` file already points to the Llama weights and tokenizer.

If you are running BF16, you may need to reduce `max_seq_length` or use a larger GPU.

In [ ]:
from dotenv import load_dotenv
load_dotenv(REPO_ROOT / '.env')

import gc
import torch

from app.llama_models import LlamaBF16, LlamaCompressed

def load_model(backend: str):
    if backend == 'bf16':
        return LlamaBF16(max_seq_length=2048, batch_size=1, device='cuda', is_batch=True)
    return LlamaCompressed(
        max_seq_length=16384,
        batch_size=1,
        device='cuda',
        is_batch=True,
        bit_width=BIT_WIDTH,
        dims=128,
    )

model = load_model(BACKEND)
print(type(model).__name__, 'loaded')
print('Allocated VRAM:', torch.cuda.memory_allocated() / (1024 ** 3), 'GB')


## 3. Build the RAG Pipeline

The benchmark uses the same modular `RAG` pipeline as the rest of the repo.
If `data/corpus.json` is not available, the notebook falls back to a tiny demo corpus so the flow still runs end-to-end.

In [ ]:
from rag_library import RAG, Chunker, VectorStore, BGEmbedder, BF16LlamaGenerator, TurboQuantLlamaGenerator

embedder = BGEmbedder(batch_size=12)
if BACKEND == 'bf16':
    generator = BF16LlamaGenerator(model=model, max_tokens=MAX_NEW_TOKENS)
else:
    generator = TurboQuantLlamaGenerator(model=model, max_tokens=MAX_NEW_TOKENS)

rag = RAG(
    embedder=embedder,
    generator=generator,
    chunker=Chunker(chunk_size=600, overlap=150, skip_noisy_pages=False),
    vector_store=VectorStore(),
    top_k=5,
)

if CORPUS_PATH.exists():
    corpus = CORPUS_PATH
    print('Loading corpus from', corpus)
else:
    corpus = [
        {'text': 'Backpropagation computes gradients by applying the chain rule backward through the network.', 'source': 'demo.pdf', 'page': 1},
        {'text': 'Convolutional neural networks use shared weights and local receptive fields to reduce parameters.', 'source': 'demo.pdf', 'page': 2},
        {'text': 'Self-attention lets each token attend to all other tokens in the sequence.', 'source': 'demo.pdf', 'page': 3},
    ]
    print('Using demo corpus fallback')

if INDEX_PATH.exists() and CHUNKS_PATH.exists():
    print('Loading prebuilt index')
    rag.load(INDEX_PATH, CHUNKS_PATH)
else:
    RESULTS_DIR.mkdir(parents=True, exist_ok=True)
    print('Building index')
    rag.build_index(corpus)
    rag.save(INDEX_PATH, CHUNKS_PATH)

print('Index size:', rag.vector_store.size)


## 4. Inspect `questions.json`

The benchmark runner loads the first `QUESTION_LIMIT` questions from the file.
If `reference_answer` is present, the result rows will also include exact match and token-level F1.

In [ ]:
from rag_library import load_questions

questions = load_questions(QUESTIONS_PATH, limit=QUESTION_LIMIT)
print('Loaded questions:', len(questions))
for q in questions:
    print(f'- {q.question_id}: {q.question}')


## 5. Run the Benchmark

Each question run records the metrics from the profiling notebook and appends a JSONL row under `results/question_benchmark/`.

In [ ]:
from rag_library import QuestionBenchmarkRunner

runner = QuestionBenchmarkRunner(
    rag=rag,
    questions_path=QUESTIONS_PATH,
    limit=QUESTION_LIMIT,
    output_dir=RESULTS_DIR,
    run_name=RUN_NAME,
    config_name=f'{BACKEND}_questions',
    max_new_tokens=MAX_NEW_TOKENS,
    use_profiler=USE_PROFILER,
)

rows = runner.run()
print('Saved to:', runner.output_file)


## 6. Summarize Results

The notebook prints a compact table of the key metrics and the aggregate means across the selected questions.

In [ ]:
import pandas as pd
from IPython.display import display
from rag_library.telemetry import load_results
from rag_library.benchmarks import summarize_measurements

df = pd.DataFrame(load_results(runner.output_file))
display(df[[
    'question_id', 'status', 'ttft_ms', 'tpot_ms', 'total_ms',
    'peak_vram_torch_mb', 'peak_vram_smi_mb', 'mean_gpu_util_pct',
    'exact_match', 'f1'
]] if len(df) else df)

summary = summarize_measurements(df.to_dict(orient='records'))
summary


## 7. Notes

- Increase `QUESTION_LIMIT` to run more questions.
- Set `USE_PROFILER = True` only when you need kernel-level traces; it adds overhead.
- Replace `questions.json` with your own question set to benchmark your corpus.
